# LSTM Model Training

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
X = np.load('../data/processed/X_sequences.npy')
y = np.load('../data/processed/y_sequences.npy')

# Stratified split to ensure failures appear in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'Train failures: {int(y_train.sum())}, Test failures: {int(y_test.sum())}')

In [ ]:
# Failure Probability = 1 / (1 + e^(-z)) — sigmoid for binary classification

# Class weights to fix recall=0 problem (imbalanced dataset)
class_weights_arr = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: float(class_weights_arr[0]), 1: float(class_weights_arr[1])}
print(f'Class weights: {class_weight_dict}')

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        'accuracy'
    ]
)
model.summary()

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('../models/motor_lstm_model.h5', save_best_only=True, monitor='val_auc', mode='max', verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
results = model.evaluate(X_test, y_test)
metric_names = ['loss', 'auc', 'recall', 'precision', 'accuracy']
for name, val in zip(metric_names, results):
    print(f'Test {name}: {val:.4f}')

print('Model saved to ../models/motor_lstm_model.h5')